# GroupBy Aggregation & Segment Insights

This notebook demonstrates segment-level aggregation using Pandas split-apply-combine workflow, multi-level groupbys, pivot tables, performance ranking, and business insight surfacing.

### Tasks Covered:
1. **Single-Level GroupBy**: Calculate segment metrics (`churn_rate`, `total_revenue`, `customer_count`, `avg_support_tickets`).
2. **Multi-Level GroupBy**: Group by `customer_type` and `product` simultaneously and unstack.
3. **Pivot Table**: Create a 2D summary matrix of revenue across customer segments and products.
4. **Rank & Identify Performers**: Rank segments by churn rate and compute percentage revenue contribution.
5. **Actionable Segment Insights**: Format segment findings and export recommendations.

## Task 1: Single-Level GroupBy with Multiple Aggregations

Group data by `customer_type` and compute churn rate, total revenue, customer count, and average support tickets.

In [ ]:
import pandas as pd
import numpy as np
import json
import os

np.random.seed(42)
n = 1000

types = np.random.choice(['Enterprise', 'SMB', 'Startup'], size=n, p=[0.05, 0.40, 0.55])
products = np.random.choice(['SaaS Platform', 'Consulting Service', 'API Access', 'Enterprise Support'], size=n)
churn_map = {'Enterprise': 0.01, 'SMB': 0.12, 'Startup': 0.08}
churns = [np.random.binomial(1, churn_map[t]) for t in types]
rev_map = {'Enterprise': (140000, 15000), 'SMB': (2500, 400), 'Startup': (1200, 200)}
revenues = [max(100.0, np.random.normal(rev_map[t][0], rev_map[t][1])) for t in types]
ticket_map = {'Enterprise': 1.5, 'SMB': 8.5, 'Startup': 5.2}
tickets = [max(0, int(np.random.poisson(ticket_map[t]))) for t in types]

df = pd.DataFrame({
    'customer_id': np.arange(1001, 1001 + n),
    'customer_type': types,
    'product': products,
    'churn': churns,
    'revenue': np.round(revenues, 2),
    'support_tickets': tickets
})

segment_metrics = df.groupby('customer_type').agg({
    'churn': 'mean',
    'revenue': 'sum',
    'customer_id': 'count',
    'support_tickets': 'mean'
})

segment_metrics.columns = ['churn_rate', 'total_revenue', 'customer_count', 'avg_support_tickets']
print(segment_metrics)

## Task 2: Multi-Level GroupBy

Group by `customer_type` and `product` simultaneously and unstack for clean display.

In [ ]:
product_segment = df.groupby(['customer_type', 'product']).agg({
    'revenue': 'sum',
    'customer_id': 'count'
})

product_segment.columns = ['total_revenue', 'customer_count']
product_segment_pivot = product_segment.unstack()
print(product_segment_pivot)

## Task 3: Pivot Table

Generate 2D matrix view of revenue using `pd.pivot_table`.

In [ ]:
pivot = pd.pivot_table(
    df,
    values='revenue',
    index='customer_type',
    columns='product',
    aggfunc='sum'
)
print(pivot)

## Task 4: Rank and Identify Top/Bottom Performers

Rank segments by churn rate and compute percentage revenue contribution.

In [ ]:
segment_metrics['churn_rank'] = segment_metrics['churn_rate'].rank()
worst_first = segment_metrics.sort_values('churn_rate', ascending=False)
print('Worst Performers First:')
print(worst_first)

segment_metrics['revenue_contribution'] = (segment_metrics['total_revenue'] / segment_metrics['total_revenue'].sum() * 100)
print('\nRevenue Contribution and Churn Rates:')
print(segment_metrics[['revenue_contribution', 'churn_rate']])

## Task 5: Surface Actionable Segment Insights

Format segment insights and save recommendations.

In [ ]:
insights = []
for segment in segment_metrics.index:
    row = segment_metrics.loc[segment]
    insight = {
        'segment': segment,
        'customer_count': int(row['customer_count']),
        'churn_rate': f"{row['churn_rate']:.1%}",
        'total_revenue': f"${row['total_revenue']:.0f}",
        'revenue_contribution': f"{row['revenue_contribution']:.1f}%",
        'action': ''
    }
    if row['churn_rate'] > 0.10:
        insight['action'] = 'HIGH PRIORITY: Churn above 10%. Investigate pain points.'
    elif row['churn_rate'] < 0.02:
        insight['action'] = 'Healthy. Maintain current service level.'
    else:
        insight['action'] = 'Monitor. No immediate action needed.'
    insights.append(insight)

insights_df = pd.DataFrame(insights)
print(insights_df.to_string(index=False))
os.makedirs('../output', exist_ok=True)
insights_df.to_csv('../output/segment_insights.csv', index=False)